# Tablespec Integration Tests on Databricks

Runs the test suite **in-process** using `ipytest` for notebook-friendly output.

Key design decisions:
- **`%pip install -e`** (magic, not subprocess) — triggers interpreter restart so `.pth`
  files are processed and `import tablespec` just works. No `sys.path` hacking.
- **`ipytest`** — thin pytest wrapper that renders results inline in the notebook.
- **In-process execution** — critical so the `spark_session` fixture can pick up the
  runtime's active SparkSession via `create_delta_spark_session() → getActiveSession()`.
  A subprocess cannot access the Databricks Spark Connect session.

**Requirements:** Attach to any Databricks cluster or use serverless compute.

In [0]:
# %pip triggers an interpreter restart, so .pth files from editable installs
# are processed automatically — no sys.path manipulation needed.
%pip install -e /Workspace/Users/erik.labianca@synaptiq.ai/tablespec --quiet
%pip install ipytest pytest-cov pytest-mock anyio hypothesis --quiet

In [0]:
import ipytest
import os
import sys

PROJECT_ROOT = "/Workspace/Users/erik.labianca@synaptiq.ai/tablespec"

# Disable bytecode caching (workspace FS doesn't support __pycache__)
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True

# Configure ipytest: notebook-friendly output, raises on failure
ipytest.autoconfig(
    addopts=["-v", "--tb=short", "-p", "no:cacheprovider"],
    run_in_thread=False,
    raise_on_error=True,
)

In [0]:
ipytest.run(os.path.join(PROJECT_ROOT, "tests/integration/"))

In [0]:
# Full suite: unit + integration (skips modules needing local-only spark setup)
ipytest.run(
    os.path.join(PROJECT_ROOT, "tests/"),
    "--ignore=tests/unit/test_quality_executor_selection.py",
    "--ignore=tests/unit/test_baseline_service.py",
)